In [1]:
%load_ext autoreload
%autoreload 2

# Tabel 010 kvantorid 1

Lisaandmetena kasutatakse skripriga 910 kokku kogutud lemma pos korpuses esinemise statistikat.

Korpusest kogutakse kokku tipud - ülemus + vahetu alluv, kus:
* ülemuse sõnaliik on `NOUN` ja kääne `part` (p);
* alluv eelneb lauses ülemusele;
* alluva sünrel on `nmod`, sõnaliik on `NOUN` ja kääne `nom` või `gen` või `part`

**Ülesande originaalpüstitus**

Otsime korpusest ülemuse-alluva paare (nt tass kohvi, tassist kohvist, tassi kohviga), kus
1. ülemuse sõnaliik = NOUN ja kääne = p (part) + alluva sünrel = nmod ja kääne = n (nom), p (part) või g (gen)

Tulemuste tabelis võiksid olla järgmised veerud: alluva lemma, alluva kääne, alluva arv, ülemuse lemma, ülemuse kääne, ülemuse arv, kogu lause, ?päringule vastav fragment, alluva lemma koguarv korpuses.

**Tulemus**

Tulemuseks on tabel tsv formaadis.


Tabeli veerud
||||
|---|---|---|
|**child_lemma**| alluva lemma |---|
|**child_case**| alluva kääne |---|
|**child_number**| alluva arv |---|
|**parent_lemma**| ülemuse lemma |---|
|**parent_case**| ülemuse käänel |---|
|**parent_number**| ülemuse arv |---|
|**text**| ?päringule vastav fragment |---|
|**sentence**| teve lause tekst, kus ülemus ja alluv toodetud esile alakriipsudega  \_\_sõne\_\_ |---|
|**sentence_id**| lause id koondkorpuse andmebaasis|---|
|**child_lemma_total**| lemma + POS esinemise arv Õpikulausete korpuses |---|


In [2]:
from notebook_context import LISTS_FOLDER, corpus_reader, clean_lemma

import pandas as pd
from datetime import datetime

FIELDNAMES = [
    "keeletase",
    "emakeel",
    "klass",
    "child_lemma",
    "child_pos",
    "child_case",
    "child_number",
    "parent_lemma",
    "parent_pos",
    "parent_case",
    "parent_number",
    "text",
    "sentence",
    "sentence_id",
    "child_lemma_total",
]

LEMMAS_STAT = LISTS_FOLDER / "stats/lemmas.tsv"
QUANTIFIERS_LIST = "./lists/110-112.quantifiers.csv"

TYPE = "quantifier_1"
date_time = datetime.now().strftime("%Y%m%d-%H%M%S")

OUT_FILE = LISTS_FOLDER / "results" / f"{TYPE}_{date_time}.tsv"


PARENT_FILTER_CASES = (
   "Par", # part
)

MOD_FILTER_CASES = (
    "Nom", # nom
    "Gen", # gen
    "Par", # part
)

In [3]:
# loeme sisse lemmade statistika ja teeme vastava dict
df_lemmas = pd.read_csv(LEMMAS_STAT, sep="\t")
lemmas_stat = {f"{row['lemma']}\t{row['POS']}": int(row['total']) for _, row in df_lemmas.iterrows()}
lemmas_stat['olema\tVERB']

8718

In [4]:
# qunitifiers
df_quantifiers = pd.read_csv(QUANTIFIERS_LIST)
my_quantifiers = list(df_quantifiers['lemma'].unique())
len(my_quantifiers)

435

In [5]:
%%time

  
collocations = []
count = 0
for sentence_id, graph in corpus_reader.get_sentences():
    count += 1
    if not sentence_id:
        sentence_id = count

    keeletase = graph.get_metadata("doc").get("keeletase")
    emakeel = graph.get_metadata("doc").get("emakeel")
    klass = graph.get_metadata("doc").get("klass")

    # matrix for node distances
    dpath = graph.get_distances_matrix()

    # noun nodes
    noun_nodes = graph.get_nodes_by_attributes(attrname="POS", attrvalue="NOUN")

    # nmod
    nmod_nodes = graph.get_nodes_by_attributes(attrname="deprel", attrvalue="nmod")

    # iteratsioon üle nimisõnade
    for noun in noun_nodes:
        noun_lemma = graph.nodes[noun]["lemma"]
        noun_pos = graph.nodes[noun]["POS"]
        noun_case = graph.get_node_case(noun)
        noun_number = graph.get_node_number(noun)

        if noun_case not in PARENT_FILTER_CASES:
            continue
        # childnodes
        kids = [k for k in dpath[noun] if dpath[noun][k] == 1]

        # iterate over nmod children
        for nmod in nmod_nodes:
          
            # kui pole vahetu alluv, siis ei huvita
            if nmod not in kids:
                continue

            # nmod peab olema lauses enne ülemust
            if nmod > noun:
                continue
            nmod_case = graph.get_node_case(nmod)
            nmod_pos = graph.nodes[nmod]["POS"]
            if nmod_case not in MOD_FILTER_CASES:
                continue
            if nmod_pos != "NOUN":
                continue

            nmod_lemma = graph.nodes[nmod]["lemma"]
            
            # kontrollime, kas nmod on meie huvipakkuvate kvantifikaatorite hulgas
            if clean_lemma(nmod_lemma) not in my_quantifiers:
                continue
            
            nmod_number = graph.get_node_number(nmod)

            words = []
            for n in sorted(graph.nodes):
                if not n:
                    continue
                if n in (noun, nmod):
                    words.append(f'___{graph.nodes[n]["form"]}___')
                else:
                    words.append(graph.nodes[n]["form"])
            sentence_text = " ".join(words)

            text = " ".join([graph.nodes[n]["form"] for n in sorted((noun, nmod))])

            collocations.append(
                (
                    keeletase,
                    emakeel,
                    klass,
                    nmod_lemma,  # child_lemma
                    nmod_pos,  # child_pos
                    nmod_case,  # child_case
                    nmod_number,  # child_number
                    noun_lemma,  # parent_lemma
                    noun_pos,  # parent_pos
                    noun_case,  # parent_case
                    noun_number,  # parent_number
                    text,  # text
                    sentence_text,  # sentence
                    sentence_id,  # sentence_id
                    lemmas_stat.get(f"{nmod_lemma}\t{nmod_pos}", 0),  # child_lemma_total
                )
            )

print(f"sentences processed: {count}")
print(f"collocations found: {len(collocations)}")

../data/vrt-with-meta-corpus-02-06-25_ordered.vrt
sentences processed: 134551
collocations found: 448
CPU times: user 16.3 s, sys: 0 ns, total: 16.3 s
Wall time: 16.4 s


In [6]:

df = pd.DataFrame(collocations, columns=FIELDNAMES)
df.to_csv(OUT_FILE, sep="\t", index=None)
df.head()

,keeletase,emakeel,klass,child_lemma,child_pos,child_case,child_number,parent_lemma,parent_pos,parent_case,parent_number,text,sentence,sentence_id,child_lemma_total
0,5,0,6,jalg,NOUN,Gen,Sing,pall,NOUN,Par,Sing,jalg palli,Sünipäev algab kell 10.00 minu kodus jaminu sü...,4532_3,150
1,5,0,6,sõõm,NOUN,Gen,Sing,kook,NOUN,Par,Sing,sõõme koogi,algab kell neli ja lõbeb kell kuus.Me seal män...,4768_4,3
2,5,0,6,pudel,NOUN,Nom,Sing,vesi,NOUN,Par,Sing,pudel vett,Võtta kaasa ___pudel___ ___vett___ ja spordiri...,5643_6,12
3,0,1,9,hulk,NOUN,Nom,Sing,inimene,NOUN,Par,Plur,hulk inimesi,"Näiteks Eesti laulupidu , kus kantakse ette ig...",6007_11,84
4,0,1,9,tund,NOUN,Gen,Plur,viis,NOUN,Par,Sing,tundide viisi,"Tavaliselt ongi nii , et muusekatunde ei naudi...",6008_9,1869
